# The limit order book

Section 1.1 of the lecture notes, at the keyboard.  The definitions are stated there; here we
build the object they describe, read the derived quantities off it, and watch one order at a
time change it.

Prices are integer counts of ticks throughout.  The conversion to currency happens once, at
the boundary, and nowhere else.

In [1]:
from pathlib import Path

import pandas as pd

from unito26.lob import frames, lobster
from unito26.lob.messages import (
    BUY, SELL, GridDepth, ReportedDepth, SweepSize,
    TickGrid, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook
from unito26.lob.visualization import (
    ascii_ladder, book_figure, describe_message, example_figure, use_template,
)
from unito26.lob.worked_examples import CATALOGUE, BASELINE_BOOK, check, to_sides

use_template()

GRID = TickGrid(0.01)
DEPTH = ReportedDepth(10)

## 1. An order is four numbers

A limit order is the tuple $(t, q, p, d)$: time, size, price, direction.  The direction is an
`int`, $+1$ to buy and $-1$ to sell, because the single expression $pd$ collapses both sides
of the book into one comparison.

In [2]:
for message in (
    limit_order(0.0, 250, 999, SELL),
    market_order(0.0, 250, SELL),
    market_order(0.0, 250, BUY),
    withdrawal(0.0, 60, 999, BUY),
):
    print(f"{describe_message(message):<22} {message.kind.name:<8} price={message.price}")

sell 250 @ 999         SUBMIT   price=999
market sell 250        SUBMIT   price=0
market buy 250         SUBMIT   price=9223372036854775807
withdraw 60 from the buy side at 999 WITHDRAW price=999


A market order is not a third kind of message.  It is a limit order carrying a price
*specification* that guarantees execution: $0$ for a sell, and the largest representable
integer for a buy.  Neither is a price at which anything may rest — which is why the buy
prints as `9223372036854775807` and why `describe_message` does not show it.

The tick grid is the boundary.  It converts currency to ticks on the way in, and it rejects a
price that is not a multiple of the tick rather than rounding it: a rounded price is a level
that fails to compare equal to the one it should have been.

In [3]:
print("10.02 ->", GRID.to_ticks(10.02), "ticks")
print("1002  ->", GRID.to_price(1002))

try:
    GRID.to_ticks(10.025)
except ValueError as error:
    print("off the grid:", error)

10.02 -> 1002 ticks
1002  -> 10.02
off the grid: price 10.025 is not a multiple of tick size 0.01


## 2. One order meets a book

The book of §1.1 is the aggregated one: a map from price to the total size resting there, and
nothing about which orders make up that total.  We use the worked example of the notes.

In [4]:
book = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))
print(ascii_ladder(book))

    1003  #########################      180  ask
    1002  #################              120  ask
          ----------------------------  spread 2
    1000  ##############                 100  bid
     999  ############################   200  bid
     998  #####################          150  bid


Now one sell order for 400 shares, priced at 999.  It is priced into the bid side, so it
trades before it rests.

In [5]:
result = book.submit(limit_order(1.0, 400, 999, SELL), record=True)

for fill in result.fills:
    print(f"filled {fill.size:>4} at {fill.price}")
print(f"q_M = {result.market_order_size}, rested = {400 - result.market_order_size}")
print()
print(ascii_ladder(book))

filled  100 at 1000
filled  200 at 999
q_M = 300, rested = 100

    1003  ############################   180  ask
    1002  ###################            120  ask
     999  ################               100  ask
          ----------------------------  spread 1
     998  #######################        150  bid


The bid side lost two whole levels, so the best bid moved from 1000 down to 998.  The 100
shares that could not be filled came to rest at 999 — on the **ask** side, a tick below the
old best bid, on a level this order had just cleared.  Every ask price therefore shifted:
1002 was the best ask and is now three ticks away from it.

Each fill printed at the **resting** order's price, never at the 999 the incoming order named.

In [6]:
book_figure(book, depth=6, title="after the sell of 400 at 999")

## 3. The grid is not a list of queues

Level $i$ is a *position on the price grid*, occupied or not.  A market data feed reports the
first $L$ prices that *carry size*.  The two agree only on a book with no holes, and the book
we just made has two.

In [7]:
print("grid positions  :", book.levels(SELL, 6))
print("occupied levels :", book.occupied_levels(SELL, ReportedDepth(6)))

grid positions  : [(999, 100), (1000, 0), (1001, 0), (1002, 120), (1003, 180), (1004, 0)]
occupied levels : [(999, 100), (1002, 120), (1003, 180)]


Three of the six grid positions read zero, and they are not the same kind of zero.  The zeros
at 1000 and 1001 sit *between* two occupied prices; the one at 1004 sits past the last of
them.  Only the first kind is a gap, which is why `empty_grid_positions` returns two indices
and not three.

In [8]:
print("empty grid positions:", book.empty_grid_positions(SELL, ReportedDepth(6)))
print("gaps                :", book.gap_count(SELL, ReportedDepth(6)))
print("occupied levels     :", book.occupied_level_count(SELL, ReportedDepth(6)))
print("grid span they cover:", book.grid_span(SELL, ReportedDepth(6)))

empty grid positions: [2, 3]
gaps                : 1
occupied levels     : 3
grid span they cover: 5


Three occupied levels spanning five grid positions.  Any quantity indexed by level means
something different under the two readings, and §1.2 is where that difference is paid for.

## 4. What one configuration yields

Spread, mid-price, micro-price, queue imbalance and sweep cost are all functions of the state
the book already holds.  None of them needs the tape.

In [9]:
fresh = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))

print("best bid / best ask:", fresh.best_bid_price, "/", fresh.best_ask_price)
print("spread             :", fresh.spread)
print("mid-price          :", fresh.mid_price)
print("micro-price        :", round(fresh.micro_price, 4))
for n in (1, 2, 3):
    print(f"queue imbalance I^{n}:", round(fresh.queue_imbalance(GridDepth(n)), 4))

best bid / best ask: 1000 / 1002
spread             : 2
mid-price          : 1001.0
micro-price        : 1000.9091
queue imbalance I^1: -0.0909
queue imbalance I^2: 0.0
queue imbalance I^3: 0.2


The imbalance is bid minus ask over the total, so a bid-heavy book is positive.  Here the ask
is heavier at the touch and $I^1$ is negative.

The micro-price has a second expression: the weighting in which each price carries the size
resting on the *opposite* side.  Computing it that way is an independent route to the same
number.

In [10]:
bid_size, ask_size = fresh.best_bid_size, fresh.best_ask_size
opposite_weighted = (bid_size * fresh.best_ask_price + ask_size * fresh.best_bid_price) / (
    bid_size + ask_size
)
print("by definition:", fresh.micro_price)
print("by weighting :", opposite_weighted)
assert abs(fresh.micro_price - opposite_weighted) < 1e-12

by definition: 1000.9090909090909
by weighting : 1000.9090909090909


### Walking the book

The sweep cost is the per-share cost of a market order, in ticks, measured against the mid.
Every share pays the half-spread; each additionally pays its own distance from the touch.

The book knows whether an order walked — `walked_the_book` asks whether the fills printed at
more than one price — so the criterion and the cost are two independent readings, and we can
put them side by side.

In [11]:
half_spread = fresh.spread / 2
print(f"half-spread: {half_spread}\n")

for size in (50, 100, 120, 200, 300):
    probe = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))
    filled = probe.submit(market_order(1.0, size, BUY), record=True)
    cost = fresh.sweep_cost(BUY, SweepSize(size), ReportedDepth(10))
    print(f"buy {size:>3}: sweep cost {cost:.4f}   walked: {filled.walked_the_book}")

half-spread: 1.0

buy  50: sweep cost 1.0000   walked: False
buy 100: sweep cost 1.0000   walked: False
buy 120: sweep cost 1.0000   walked: False
buy 200: sweep cost 1.4000   walked: True
buy 300: sweep cost 1.6000   walked: True


The two columns agree, and they agree at the boundary: 120 clears the touch exactly, prints at
one price, and costs exactly the half-spread.  An order walks the book precisely when its
sweep cost exceeds the half-spread — the walk term being positive exactly when some share is
taken at a level past the touch.

The equivalence holds only where the sweep cost is defined, and it is not defined for an order
larger than the side.

In [12]:
ask_side = sum(size for _, size in fresh.occupied_levels(SELL, ReportedDepth(10)))
print("ask side holds   :", ask_side)
print("sweep cost of 400:", fresh.sweep_cost(BUY, SweepSize(400), ReportedDepth(10)))

ask side holds   : 300
sweep cost of 400: None


`None`, and not a number.  The quantity is defined only where the side holds at least the
shares asked for, and code must branch on that rather than return something plausible.

## 5. Every limit order is a market order and a resting order

Processing $(t,q,p,-1)$ is processing the pair $[(t,q_M,0,-1), (t,q-q_M,p,-1)]$: the market
part, then the resting part.  The book reports $q_M$ directly, so we can check it against the
definition read off the opposite side.

In [13]:
def price_eligible_size(book, price, direction):
    '''Resting size on the far side that an order at ``price`` may take.'''
    opposite = book.levels_map(-direction)
    return sum(
        resting
        for level, resting in opposite.items()
        if level * direction <= price * direction
    )


probe = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))
predicted = min(400, price_eligible_size(probe, price=999, direction=SELL))
executed = probe.submit(limit_order(1.0, 400, 999, SELL), record=True).market_order_size

print("q_M by definition:", predicted)
print("q_M by the engine:", executed)

q_M by definition: 300
q_M by the engine: 300


A matching engine therefore needs **one** code path.  The loop that consumes the opposite side
simply does not execute when nothing matches, so a passive order takes the same route as an
aggressive one.

In [14]:
passive = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))
joined = passive.submit(limit_order(2.0, 75, 999, BUY), record=True)

print("fills       :", joined.fills)
print("q_M         :", joined.market_order_size)
print("bid side now:", passive.levels_map(BUY))

fills       : []
q_M         : 0
bid side now: {1000: 100, 999: 275, 998: 150}


No fills, one code path, and the 75 shares joined the queue already resting at 999.

## 6. The catalogue

Eleven transitions, one for each branch of the update rule.  Having one example per branch is
what makes it possible to argue the set is complete rather than merely plausible.  `check`
asserts the resulting state, which is derived from the notation rather than captured from a
run.

In [15]:
for example in CATALOGUE:
    check(AggregateBook, example)
    print(f"  {example.name}")
print(f"\n{len(CATALOGUE)} examples, all passing")

  a passive buy joins an occupied level
  a passive buy rests inside the spread
  a sell takes part of the best bid
  a sell clears the best bid exactly
  a sell executed in full, with no remainder
  a sell that walks the book and rests the remainder
  a sell consumes the whole bid side
  a market sell larger than the book
  a market buy into an empty ask side
  a withdrawal away from the best
  a withdrawal that empties the best bid

11 examples, all passing


In [16]:
example_figure(next(e for e in CATALOGUE if "case B" in e.name), depth=7)

## 7. Two things the aggregated state cannot answer

**Whose size it is.**  A sum does not record its summands.  Fold two different streams of
orders — one submission of 100, or five of 20 — and the book that results is the same book.

In [17]:
one_order = [limit_order(1.0, 100, 1000, BUY)]
five_orders = [limit_order(1.0 + i, 20, 1000, BUY) for i in range(5)]

left, right = AggregateBook(), AggregateBook()
for stream, target in ((one_order, left), (five_orders, right)):
    for message in stream:
        target.apply(message, record=False)

print("one submission of 100:", left.levels_map(BUY))
print("five submissions of 20:", right.levels_map(BUY))
print("same state:", left.levels_map(BUY) == right.levels_map(BUY))

one submission of 100: {1000: 100}
five submissions of 20: {1000: 100}
same state: True


So no question that names an order can be answered from this state: how much size is ahead of
mine, will my order be filled, whose fill was that, what is one account's position.  Those
need the order-level book, which carries identity and which we do not build here.

**How much traded.**  Two books can reach the same state by different routes, one of which
traded and one of which did not.

In [18]:
executed, cancelled = (AggregateBook.from_levels(*to_sides(BASELINE_BOOK)) for _ in range(2))

executed.apply(market_order(1.0, 40, SELL), record=False)
cancelled.apply(withdrawal(1.0, 40, 1000, BUY), record=False)

print("after a market sell of 40:", executed.levels_map(BUY), "traded", executed.last_traded_size)
print("after a withdrawal of 40 :", cancelled.levels_map(BUY), "traded", cancelled.last_traded_size)
print("same state:", executed.levels_map(BUY) == cancelled.levels_map(BUY))

after a market sell of 40: {1000: 60, 999: 200, 998: 150} traded 40
after a withdrawal of 40 : {1000: 60, 999: 200, 998: 150} traded 0
same state: True


Same state, and 40 shares traded against none.  A level shrinks by cancellation as well as by
execution, so **no sequence of sizes determines the volume traded** — which is why volume is
read off the tape, as $\sum q_M$ over the market orders of a window, and never differenced out
of the book.

What survives is exactly what this chapter is for: how size has accumulated across the levels
of the two sides.

## 8. A real book, and its padding

A LOBSTER orderbook file is $4L$ columns — ask price, ask size, bid price, bid size, repeated
outward from the touch — and no timestamp of its own.  One row is one configuration.

In [19]:
SAMPLE = Path("../../data/lobster/AMZN_2012-06-21_34200000_57600000_orderbook_10.csv")
UNIT = lobster.price_unit(GRID)

# The first row of AMZN on 2012-06-21, so that the notebook runs with no data at all.
FIRST_ROW = (
    2239500, 100, 2231800, 100, 2239900, 100, 2230700, 200, 2240000, 220,
    2230400, 100, 2242500, 100, 2230000, 10, 2244000, 547, 2226200, 100,
    2245400, 100, 2213000, 4000, 2248900, 100, 2204000, 100, 2267700, 100,
    2202500, 5000, 2294300, 100, 2202000, 100, 2298000, 100, 2189700, 100,
)

if SAMPLE.exists():
    snapshots = lobster.load_orderbook(SAMPLE, DEPTH)
    print(f"read {len(snapshots):,} snapshots from {SAMPLE.name}")
else:
    snapshots = pd.DataFrame([FIRST_ROW], columns=lobster.orderbook_columns(DEPTH))
    print("sample file absent; using the single row carried in this notebook")

row = snapshots.iloc[0]
print("price unit:", UNIT, "file units per tick")

read 269,748 snapshots from AMZN_2012-06-21_34200000_57600000_orderbook_10.csv
price unit: 100 file units per tick


In [20]:
amzn = AggregateBook.from_lobster_row(row, UNIT, DEPTH)

print("best bid / best ask:", amzn.best_bid_price, "/", amzn.best_ask_price, "ticks")
print("                   :", round(GRID.to_price(amzn.best_bid_price), 2), "/",
      round(GRID.to_price(amzn.best_ask_price), 2), "dollars")
print("spread             :", amzn.spread, "ticks")
print("mid-price          :", amzn.mid_price, "ticks")
print("I^1                :", round(amzn.queue_imbalance(GridDepth(1)), 4))
print()
print("occupied ask levels:", amzn.occupied_levels(SELL, DEPTH)[:4])
print("grid span, ask side:", amzn.grid_span(SELL, DEPTH), "positions for", DEPTH, "levels")
print("gaps, ask side     :", amzn.gap_count(SELL, DEPTH))

best bid / best ask: 22318 / 22395 ticks
                   : 223.18 / 223.95 dollars
spread             : 77 ticks
mid-price          : 22356.5 ticks
I^1                : 0.0

occupied ask levels: [(22395, 100), (22399, 100), (22400, 220), (22425, 100)]
grid span, ask side: 586 positions for 10 levels
gaps, ask side     : 8


The spread is odd, so the mid sits on a half tick.  `to_price` takes an integer count of
ticks and would refuse it: the mid-price is a derived quantity, not a price at which anything
can rest or trade, and the grid is under no obligation to carry it.

Ten reported levels spanning several hundred grid positions.  This is a small-tick instrument,
and the distinction of §3 is not a corner case on it: $I^2$ read off the columns of this file
pairs the touch with a price some way from it, and is not the $I^2$ of the notes.

The row round-trips.

In [21]:
assert amzn.to_lobster_row(UNIT, DEPTH) == row.tolist()
print("round trip holds over", len(row), "columns")

round trip holds over 40 columns


### The padding sentinels

Where a side has fewer than $L$ occupied levels, LOBSTER pads.  The two sentinels have
**opposite signs**, so the obvious filter — keep the positive prices — removes one of them and
keeps the other.  `frames` states the layout once, so we do not restate the stride by hand.

In [22]:
print("padding row at depth 2:", frames.lobster_padding_row(ReportedDepth(2)))

padded = row.copy()
padded[-4:] = frames.lobster_padding_row(ReportedDepth(1))

prices = padded[[name for name in padded.index if "Price" in name]]
sentinels = prices[prices.abs() == frames.ASK_PADDING]
print("\nsentinels present:", list(sentinels.index))
print("kept by `> 0`    :", list(sentinels[sentinels > 0].index))

padding row at depth 2: [ 9999999999           0 -9999999999           0  9999999999           0
 -9999999999           0]

sentinels present: ['AskPrice10', 'BidPrice10']
kept by `> 0`    : ['AskPrice10']


`AskPrice10` survives a filter written to remove padding, and it is worth $10^{10}$.  A
statistic computed over such a frame comes out in the billions, and nothing raises.

In [23]:
spread_in_units = padded["AskPrice1"] - padded["BidPrice1"]
bad_spread = padded["AskPrice10"] - padded["BidPrice10"]
print(f"spread at the touch : {spread_in_units:>14,} file units")
print(f"spread at level 10  : {bad_spread:>14,} file units")

spread at the touch :          7,700 file units
spread at level 10  : 19,999,999,998 file units


In [24]:
thin = AggregateBook.from_lobster_row(padded, UNIT, DEPTH)
print("occupied ask levels:", len(thin.occupied_levels(SELL, DEPTH)))
print("occupied bid levels:", len(thin.occupied_levels(BUY, DEPTH)))
print("spread still       :", thin.spread, "ticks")

occupied ask levels: 9
occupied bid levels: 9
spread still       : 77 ticks


Nine levels on each side rather than ten, and the spread unaffected: the sentinel carries size
zero, and a level of size zero is removed rather than stored.  That is the invariant doing the
work, not a special case for padding.

Whether a given file shows padding at all is a property of the instrument and the day.

In [25]:
if len(snapshots) > 1:
    sentinel_rows = (snapshots.abs() == frames.ASK_PADDING).any(axis=1).sum()
    print(f"{sentinel_rows:,} of {len(snapshots):,} rows carry a sentinel")
else:
    print("one row only; the census needs the sample file")

0 of 269,748 rows carry a sentinel


None, on this instrument at this depth on this day.  The format permits padding; this file
does not use it, and a pipeline that has only ever seen this file has never exercised the
branch that handles it.

---

**What to take away.**  The aggregated state is closed under the arrival of an order, so it
reproduces the whole public book.  Every limit order is a market part followed by a resting
part, which is one code path and not two.  A level index is a position on the grid, and a feed
reports occupied prices.  And two questions this state cannot answer — whose size it is, and
how much traded — are the two that the rest of the chapter has to work around.

Next: where the sequence of configurations comes from, and why a participant computes it
rather than receiving it.